In [1]:
import kagglehub

kagglehub.login()

In [2]:
import kagglehub

# Download latest version
path = kagglehub.competition_download('m5-forecasting-accuracy')

print("Path to competition files:", path)

import os
for dirname, _, filenames in os.walk('/root/.cache/kagglehub/competitions/m5-forecasting-accuracy'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

Path to competition files: /root/.cache/kagglehub/competitions/m5-forecasting-accuracy
/root/.cache/kagglehub/competitions/m5-forecasting-accuracy/sales_train_validation.csv
/root/.cache/kagglehub/competitions/m5-forecasting-accuracy/sell_prices.csv
/root/.cache/kagglehub/competitions/m5-forecasting-accuracy/calendar.csv
/root/.cache/kagglehub/competitions/m5-forecasting-accuracy/sample_submission.csv
/root/.cache/kagglehub/competitions/m5-forecasting-accuracy/sales_train_evaluation.csv


In [3]:
# # all path 
# import os
# os.listdir('C:\\Users\\Debasish Das\\Desktop\\Time_series\\Welmart_Sales_Forcasting\\data')


### 1. Load Dataset

In [4]:
import pandas as pd
calendar_df = pd.read_csv("/root/.cache/kagglehub/competitions/m5-forecasting-accuracy/sales_train_validation.csv")
sell_prices_df = pd.read_csv("/root/.cache/kagglehub/competitions/m5-forecasting-accuracy/sell_prices.csv")
sales_train_evaluation_df = pd.read_csv("/root/.cache/kagglehub/competitions/m5-forecasting-accuracy/sales_train_evaluation.csv")
sales_train_validation_df = pd.read_csv("/root/.cache/kagglehub/competitions/m5-forecasting-accuracy/sales_train_validation.csv")

In [5]:
calendar_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30490 entries, 0 to 30489
Columns: 1919 entries, id to d_1913
dtypes: int64(1913), object(6)
memory usage: 446.4+ MB


In [6]:
sell_prices_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6841121 entries, 0 to 6841120
Data columns (total 4 columns):
 #   Column      Dtype  
---  ------      -----  
 0   store_id    object 
 1   item_id     object 
 2   wm_yr_wk    int64  
 3   sell_price  float64
dtypes: float64(1), int64(1), object(2)
memory usage: 208.8+ MB


In [7]:
sales_train_evaluation_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30490 entries, 0 to 30489
Columns: 1947 entries, id to d_1941
dtypes: int64(1941), object(6)
memory usage: 452.9+ MB


In [8]:
sales_train_validation_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30490 entries, 0 to 30489
Columns: 1919 entries, id to d_1913
dtypes: int64(1913), object(6)
memory usage: 446.4+ MB


### **Review of Dataset**
- Reduced the file size useing Downcast of datatypes

In [9]:
import numpy as np
import pandas as pd


def downcast_dtypes(df):
    """Reduces memory usage of a pandas DataFrame by downcasting numeric types.

    Converts float64/float32 to float16 and int64/int32 to int16 (or int8 if
    possible).
    """
    start_mem = df.memory_usage().sum() / 1024**2
    print(f"Original Memory Usage: {start_mem:.2f} MB")

    for col in df.columns:
        col_type = df[col].dtype

        # Handle Integer types
        if np.issubdtype(col_type, np.integer):
            # Find the minimum and maximum value in the column
            c_min = df[col].min()
            c_max = df[col].max()

            # Downcast to int8 or int16 based on value ranges
            if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                df[col] = df[col].astype(np.int8)
            elif (
                c_min > np.iinfo(np.int16).min
                and c_max < np.iinfo(np.int16).max
            ):
                df[col] = df[col].astype(np.int16)
            elif (
                c_min > np.iinfo(np.int32).min
                and c_max < np.iinfo(np.int32).max
            ):
                df[col] = df[col].astype(np.int32)

        # Handle Float types
        elif np.issubdtype(col_type, np.floating):
            c_min = df[col].min()
            c_max = df[col].max()

            # Downcast to float16 if it fits within the boundaries
            if (
                c_min > np.finfo(np.float16).min
                and c_max < np.finfo(np.float16).max
            ):
                df[col] = df[col].astype(np.float16)
            else:
                df[col] = df[col].astype(np.float32)

    end_mem = df.memory_usage().sum() / 1024**2
    print(f"Optimized Memory Usage: {end_mem:.2f} MB")
    print(
        f"Decreased by: {100 * (start_mem - end_mem) / start_mem:.1f}%"
    )

    return df


In [10]:
calendar_df = downcast_dtypes(calendar_df)
sell_prices_df = downcast_dtypes(sell_prices_df)
sales_train_evaluation_df = downcast_dtypes(sales_train_evaluation_df)
sales_train_validation_df = downcast_dtypes(sales_train_validation_df)

Original Memory Usage: 446.40 MB
Optimized Memory Usage: 95.00 MB
Decreased by: 78.7%
Original Memory Usage: 208.77 MB
Optimized Memory Usage: 130.48 MB
Decreased by: 37.5%
Original Memory Usage: 452.91 MB
Optimized Memory Usage: 96.13 MB
Decreased by: 78.8%
Original Memory Usage: 446.40 MB
Optimized Memory Usage: 95.00 MB
Decreased by: 78.7%


###  ObjectiveGoal: Generate point forecasts for daily unit sales for 28 days into the future.
- Target Variables: 28 individual columns named F1 to F28 representing daily sales volumes for each item-store combination.

## 1. LSTM model for forcasting

#### a. Creating Dataset of LSTM

In [11]:
train_lstm=sales_train_evaluation_df

In [12]:
train_lstm.sample(n=5)

,id,item_id,dept_id,cat_id,store_id,state_id,d_1,d_2,d_3,d_4,...,d_1932,d_1933,d_1934,d_1935,d_1936,d_1937,d_1938,d_1939,d_1940,d_1941
2807,FOODS_3_583_CA_1_evaluation,FOODS_3_583,FOODS_3,FOODS,CA_1,CA,0,0,0,0,...,4,1,3,0,1,0,2,0,1,2
17540,FOODS_3_071_TX_2_evaluation,FOODS_3_071,FOODS_3,FOODS,TX_2,TX,0,0,0,0,...,0,0,0,0,0,2,1,3,6,0
16710,HOUSEHOLD_2_370_TX_2_evaluation,HOUSEHOLD_2_370,HOUSEHOLD_2,HOUSEHOLD,TX_2,TX,0,1,1,2,...,0,0,0,0,0,0,0,0,0,0
26002,HOUSEHOLD_2_515_WI_2_evaluation,HOUSEHOLD_2_515,HOUSEHOLD_2,HOUSEHOLD,WI_2,WI,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
29848,FOODS_3_183_WI_3_evaluation,FOODS_3_183,FOODS_3,FOODS,WI_3,WI,0,0,0,0,...,0,1,2,1,2,0,1,3,1,1


In [13]:
train_lstm=train_lstm.T

In [14]:
train_lstm

,0,1,2,3,4,5,6,7,8,9,...,30480,30481,30482,30483,30484,30485,30486,30487,30488,30489
id,HOBBIES_1_001_CA_1_evaluation,HOBBIES_1_002_CA_1_evaluation,HOBBIES_1_003_CA_1_evaluation,HOBBIES_1_004_CA_1_evaluation,HOBBIES_1_005_CA_1_evaluation,HOBBIES_1_006_CA_1_evaluation,HOBBIES_1_007_CA_1_evaluation,HOBBIES_1_008_CA_1_evaluation,HOBBIES_1_009_CA_1_evaluation,HOBBIES_1_010_CA_1_evaluation,...,FOODS_3_818_WI_3_evaluation,FOODS_3_819_WI_3_evaluation,FOODS_3_820_WI_3_evaluation,FOODS_3_821_WI_3_evaluation,FOODS_3_822_WI_3_evaluation,FOODS_3_823_WI_3_evaluation,FOODS_3_824_WI_3_evaluation,FOODS_3_825_WI_3_evaluation,FOODS_3_826_WI_3_evaluation,FOODS_3_827_WI_3_evaluation
item_id,HOBBIES_1_001,HOBBIES_1_002,HOBBIES_1_003,HOBBIES_1_004,HOBBIES_1_005,HOBBIES_1_006,HOBBIES_1_007,HOBBIES_1_008,HOBBIES_1_009,HOBBIES_1_010,...,FOODS_3_818,FOODS_3_819,FOODS_3_820,FOODS_3_821,FOODS_3_822,FOODS_3_823,FOODS_3_824,FOODS_3_825,FOODS_3_826,FOODS_3_827
dept_id,HOBBIES_1,HOBBIES_1,HOBBIES_1,HOBBIES_1,HOBBIES_1,HOBBIES_1,HOBBIES_1,HOBBIES_1,HOBBIES_1,HOBBIES_1,...,FOODS_3,FOODS_3,FOODS_3,FOODS_3,FOODS_3,FOODS_3,FOODS_3,FOODS_3,FOODS_3,FOODS_3
cat_id,HOBBIES,HOBBIES,HOBBIES,HOBBIES,HOBBIES,HOBBIES,HOBBIES,HOBBIES,HOBBIES,HOBBIES,...,FOODS,FOODS,FOODS,FOODS,FOODS,FOODS,FOODS,FOODS,FOODS,FOODS
store_id,CA_1,CA_1,CA_1,CA_1,CA_1,CA_1,CA_1,CA_1,CA_1,CA_1,...,WI_3,WI_3,WI_3,WI_3,WI_3,WI_3,WI_3,WI_3,WI_3,WI_3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
d_1937,0,0,0,1,0,0,1,5,0,1,...,3,6,3,0,0,1,0,1,0,0
d_1938,3,0,2,3,0,0,0,4,0,1,...,1,4,3,1,2,0,1,0,1,2
d_1939,3,0,3,0,2,5,1,1,0,0,...,3,4,3,1,1,0,0,1,1,2
d_1940,0,0,0,2,1,2,1,40,1,0,...,0,1,0,0,3,1,1,0,1,5


In [15]:
train_lstm_df=train_lstm[6:]

In [16]:
train_lstm_df

,0,1,2,3,4,5,6,7,8,9,...,30480,30481,30482,30483,30484,30485,30486,30487,30488,30489
d_1,0,0,0,0,0,0,0,12,2,0,...,0,14,1,0,4,0,0,0,0,0
d_2,0,0,0,0,0,0,0,15,0,0,...,0,11,1,0,4,0,0,6,0,0
d_3,0,0,0,0,0,0,0,0,7,1,...,0,5,1,0,2,2,0,0,0,0
d_4,0,0,0,0,0,0,0,0,3,0,...,0,6,1,0,5,2,0,2,0,0
d_5,0,0,0,0,0,0,0,0,0,0,...,0,5,1,0,2,0,0,2,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
d_1937,0,0,0,1,0,0,1,5,0,1,...,3,6,3,0,0,1,0,1,0,0
d_1938,3,0,2,3,0,0,0,4,0,1,...,1,4,3,1,2,0,1,0,1,2
d_1939,3,0,3,0,2,5,1,1,0,0,...,3,4,3,1,1,0,0,1,1,2
d_1940,0,0,0,2,1,2,1,40,1,0,...,0,1,0,0,3,1,1,0,1,5


- For unifrom Distribution make all dataset in between 0-1

In [17]:
from sklearn.preprocessing import MinMaxScaler
scaler=MinMaxScaler(feature_range=(0,1))
train_lstm_df_scaled=scaler.fit_transform(train_lstm_df)

In [18]:
train_lstm_df_scaled

array([[0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.3       , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       ...,
       [0.6       , 0.        , 0.5       , ..., 0.05      , 0.08333333,
        0.16666667],
       [0.        , 0.        , 0.        , ..., 0.        , 0.08333333,
        0.41666667],
       [0.2       , 0.        , 0.16666667, ..., 0.1       , 0.        ,
        0.08333333]])

- Creating dataset like that 14 days values we geting to predict 15 day sales
- d_1--d_14-->d_15,d_2--d_15-->d_16,...

In [19]:
timestaps=14
x_train=[]
y_train=[]
for i in range(timestaps,train_lstm_df_scaled.shape[0]):
    x_train.append(train_lstm_df_scaled[i-timestaps:i])
    y_train.append(train_lstm_df_scaled[i])

In [20]:
x_train=np.array(x_train)
y_train=np.array(y_train)

In [21]:
x_train.shape,y_train.shape

((1927, 14, 30490), (1927, 30490))

#### b. Train LSTM Model

In [ ]:
from tensorflow import keras
from tensorflow.keras import layers

model=keras.Sequential()
model.add(layers.LSTM(64,activation='relu',input_shape=(x_train.shape[1],x_train.shape[2])))
model.add(layers.Dense(32,activation='relu'))
model.add(layers.Dense(30490))

model.compile(loss='mse',optimizer='adam')
model.summary()


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 64)             │     7,822,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 30490)          │     1,006,170 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 8,830,330 (33.69 MB)

 Trainable params: 8,830,330 (33.69 MB)

 Non-trainable params: 0 (0.00 B)

: 

In [ ]:
model.fit(x_train,y_train,epochs=25,batch_size=8)

#### c. test model

In [ ]:
from tensorflow import keras
loaded_model = keras.models.load_model('Welmart_Sales_Forcasting\\model\\m5_lstm_model.keras')

In [ ]:
inputs=train_lstm_df_scaled[-timestaps:]
inputs=scaler.transform(inputs)

In [ ]:
x_test=[]
x_test.append(inputs[0:timestaps])
x_test=np.array(x_test)

In [ ]:
predictions=loaded_model.predict(x_test)

In [ ]:
predictions